# 14주차 — 함정·보안·비정형
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다.
어떤 셀이 오류로 막히거나 데이터가 엉켰다면, 아무 셀에서 `reset_db()`를 실행하면 처음 상태로 돌아갑니다.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, shutil, sqlite3, urllib.request
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/data-v1/mini_emr/mini_emr.sqlite"
DB_PATH = "mini_emr.sqlite"
WORK_PATH = DB_PATH   # 실습에 사용할 파일

con = None

def reset_db():
    """DB를 원본 상태로 되돌리고 다시 연결 (막히면 언제든 호출)"""
    global con
    if con is not None:
        con.close()
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
    except Exception as e:
        if not os.path.exists(DB_PATH):
            raise RuntimeError(
                "DB 다운로드 실패 — 왼쪽 파일 탭에 mini_emr.sqlite를 업로드한 뒤 이 셀을 다시 실행하세요."
            ) from e
        print("다운로드 실패 — 업로드되어 있는 파일을 사용합니다.")
    if WORK_PATH != DB_PATH:
        shutil.copyfile(DB_PATH, WORK_PATH)   # 원본 보존, 실습은 사본에서
    con = sqlite3.connect(WORK_PATH)
    con.execute("PRAGMA foreign_keys = ON")   # SQLite는 기본으로 꺼져 있음 — 반드시 켠다
    assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1

def q(sql):
    """조회(SELECT) 실행 → 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """데이터/구조를 바꾸는 SQL(INSERT/UPDATE/DELETE/CREATE 등) 실행.
    여러 문장을 순서대로 실행하고 즉시 커밋합니다 — 트랜잭션(BEGIN/ROLLBACK) 실습은
    8주차처럼 con.cursor()를 직접 사용하세요."""
    con.executescript(sql); con.commit(); print("실행 완료")

reset_db()
q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 함정 ① 단위 불일치

In [ ]:
# 혈당을 낮은 값부터 — 이 7.8은 저혈당인가?
q("""
SELECT lab_id, test_name, result_value, unit
FROM lab WHERE test_name = 'Glucose'
ORDER BY result_value LIMIT 5
""")

In [ ]:
# 단위 감사 쿼리 — 분석 전 사전 점검
q("""
SELECT test_name, unit, COUNT(*) AS 건수,
       MIN(result_value) AS 최소, MAX(result_value) AS 최대
FROM lab GROUP BY test_name, unit ORDER BY test_name
""")

## 함정 ② 분모 — '검사받은 사람만'의 평균

In [ ]:
# 당뇨 환자 수 vs 그중 HbA1c 검사자 수
q("""
SELECT
  (SELECT COUNT(DISTINCT v.patient_id) FROM visit v
   JOIN diagnosis d ON d.visit_id = v.visit_id
   WHERE d.kcd_code LIKE 'E11%') AS 당뇨환자수,
  (SELECT COUNT(DISTINCT v.patient_id) FROM visit v
   JOIN lab l ON l.visit_id = v.visit_id
   WHERE l.test_name = 'HbA1c'
     AND v.patient_id IN (
       SELECT v2.patient_id FROM visit v2
       JOIN diagnosis d2 ON d2.visit_id = v2.visit_id
       WHERE d2.kcd_code LIKE 'E11%')) AS 당뇨환자중검사자수
""")

## 함정 ③ 비정형 텍스트의 한계

In [ ]:
# 판독문 LIKE 검색 — '경색'이 들어간 보고서
q("""
SELECT img_id, modality, body_part, report
FROM imaging WHERE report LIKE '%경색%'
""")

⚠️ "급성 경색 **소견 없음**" 같은 부정 표현도 걸립니다 — 텍스트의 의미는 LIKE가 이해하지 못합니다.

## 연습문제

In [ ]:
# [연습 Q1] mmol/L로 입력된 혈당을 mg/dL로 환산하면 얼마인가? (×18) — SELECT로 환산값을 계산해 보라
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] 채혈시각(collected_at)의 시(hour) 분포를 구하라 — 오후 채혈이 섞여 있는가?
# 힌트: strftime('%H', collected_at)
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 각 검사(test_name)의 결측(result_value IS NULL) 건수를 세어 보라
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] 판독문에 '없음'이 포함된 영상 보고서를 찾아, LIKE 검색의 함정을 확인하라
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 단위 환산**
```sql
-- ×18은 포도당에만 유효한 환산 — 검사명 조건을 반드시 함께
SELECT lab_id, result_value AS mmol, ROUND(result_value * 18, 0) AS mgdl
FROM lab WHERE test_name = 'Glucose' AND unit = 'mmol/L';
```

**A2. 채혈 시각 분포**
```sql
SELECT strftime('%H', collected_at) AS 시, COUNT(*) FROM lab GROUP BY 시 ORDER BY 시;
```

**A3. 검사별 결측**
```sql
SELECT test_name, SUM(CASE WHEN result_value IS NULL THEN 1 ELSE 0 END) AS 결측 FROM lab GROUP BY test_name;
```

**A4. 부정 표현**
```sql
SELECT img_id, report FROM imaging WHERE report LIKE '%없음%';
```